In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from helper_functions import gen_weight

model = load_model('model_gru.h5')

model.summary()

2026-06-01 12:58:13.729419: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-01 12:58:13.731479: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-01 12:58:13.760638: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-01 12:58:13.760660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-01 12:58:13.762131: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (GRU)                (None, 20)                1680      
                                                                 
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 relu_0 (Activation)         (None, 64)                0         
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
Total params: 3089 (12.07 KB)
Trainable params: 3089 (12.07 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [2]:
# import necessary inputs

import numpy as np
import os

# gru package sv file folder
gru_pkg_folder = './weights/gru_weights_biases_pkgs'
if not os.path.exists(gru_pkg_folder):
    os.makedirs(gru_pkg_folder)


def flip_bits(val, bits):
    return ((val ^ (2**bits - 1)) + 1)

def sigmoid(x):
    return 1/(1+(np.e**(-x)))

# really inefficient but calling "sed" subprocess didn't work
# removes negative sign "-" from file
def remove_negs(tmp_file, dest_file_name):
    output_file = open(tmp_file, 'r')
    dest_file = open(dest_file_name, 'w')
    file = output_file.read()
    file = file.replace('b-', 'b')
    dest_file.write(file)
    # close files
    dest_file.close()
    output_file.close()
    
# =========================================================================
# pkg functions (for sub-processes)
# =========================================================================
# pkg name: [name]_[width]_[nint]
def get_pkg_name(name, width, nfrac):
    return str(name) + '_' + str(width) + '_' + str(width - nfrac)

# def conv_to_str(num, width, nfrac):

#     # shift the decimal point to the right
#     div = (2**nfrac)*1.0
#     x = round(float(num * div // 1))

#     if (x < 0):
#         return format(flip_bits(x, width), '0'+ str(width) + 'b')
#     return format(x, '0'+ str(width) + 'b')

def conv_to_str(num, width, nfrac):
    return dec_to_bin(number=num*(2**nfrac), bits=width)

def dec_to_bin(number : int | float, bits=-1):
    """
    Converts a decimal number to a str binary representation
    
    :param number: Number to convert into binary
    :type number: int | float
    :param bits: Bitwidth of output number. If negative, uses the minimum amount
    """
    # Determines if a number is negative
    neg=False
    if (number<0):
        number*=-1
        number-=1
        neg=True
    
    # Rounds number to nearest int
    number=int(np.round(number, 0))
    out=""

    # Return maximum positive or negative number if the input cannot be represented by the bitwidth
    if (bits>0 and number>2**(bits-1)):
        return "0" + "1"*(bits-1)
    elif (bits>0 and number<(-1)*2**(bits-1)):
        return "1" + "0"*(bits-1)
    
    # Do typical conversion of decimal to binary number
    while (number>0):
        res = number%2
        if (neg):
            res= 0 if (res==1) else 1
        out=f"{res}{out}"
        if (len(out)==(bits-1)):
            break
        number=int(number/2)
        
    # Add signed bit
    if (neg):
        out=f"{1}{out}"
    else: out=f"{0}{out}"

    # If the number was 0
    if (len(out)==0):
        out="0"

    # Sign extension
    while (len(out)<bits):
        out=f"{out[0]}{out}"

    return out


# =========================================================================

In [3]:
# extract weights and biases for GRU layer (separated into 4 dense) + print to files gate_weights.txt and gate_biases.txt
for layer in model.layers:
    print(layer.name)
    if layer.name == 'layer1':

        print(layer.get_config())

        weights = layer.get_weights()
        print("Number of weights tensors: " + str(len(weights)))

        # W for x, U for h, B for biases
        W = weights[0]
        U = weights[1]
        B = weights[2]

        print("input size " + str(W.shape[0]))
        print("hidden state size " + str(layer.units))

        print("W shape: " + str(W.shape))
        print("U shape: " + str(W.shape))
        print("B shape: " + str(W.shape))

        # =========================================================================
        # extract four weights and biases for the four dense layers
        # =========================================================================

        # combine weights for reset/update gates
        update_gate_weight = np.concatenate((W[:, :layer.units], U[:, :layer.units]), axis = 0)
        reset_gate_weight = np.concatenate((W[:, layer.units:2*layer.units], U[:, layer.units:2*layer.units]), axis=0)

        # keep separate for candidate gate
        candidate_gate_W = W[:, 2*layer.units:]
        candidate_gate_U = U[:, 2*layer.units:]

        # combine input biases and recurrent biases for reset/update gates
        # note that recurrent biases are used when reset_after config set to True for the gru layer (default is True)
        update_gate_bias = np.sum(B[:, :layer.units], axis = 0)
        reset_gate_bias = np.sum(B[:, layer.units:2*layer.units], axis = 0)

        # keep separate for candidate gate
        candidate_gate_bias = B[0, 2*layer.units:]
        candidate_gate_bias_rec = B[1, 2*layer.units:]
        

        # write to npy files
        np.savetxt('reset_gate_weights.txt', reset_gate_weight.flatten())
        np.savetxt('update_gate_weights.txt', update_gate_weight.flatten())
        np.savetxt('candidate_gate_weights_W.txt', candidate_gate_W.flatten())
        np.savetxt('candidate_gate_weights_U.txt', candidate_gate_U.flatten())

        np.savetxt('reset_gate_biases.txt', reset_gate_bias.flatten())
        np.savetxt('update_gate_biases.txt', update_gate_bias.flatten())
        np.savetxt('candidate_gate_biases_W.txt', candidate_gate_bias.flatten())
        np.savetxt('candidate_gate_biases_U.txt', candidate_gate_bias_rec.flatten())

layer1
{'name': 'layer1', 'trainable': True, 'dtype': 'float32', 'batch_input_shape': (None, 20, 6), 'return_sequences': False, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'time_major': False, 'units': 20, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'VarianceScaling', 'config': {'scale': 1.0, 'mode': 'fan_in', 'distribution': 'truncated_normal', 'seed': None}, 'registered_name': None}, 'recurrent_initializer': {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'gain': 1.0, 'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': {'module': 'keras.regularizers', 'class_name': 'L1L2', 'config': {'l1': 9.999999747378752e-06, 'l2': 9.999999747378752e-05}, 'registered_name': None}, 'recurrent_regularizer': None, 'b

In [4]:
# parameters - alter this to reflect the state of model
INPUT_SIZE_GRU = 6      # size of x_t / feature size (length per time step)
OUTPUT_SIZE_GRU = 20   # size of h_t / final y_t

In [5]:
# =======================================================
# generate dense weights and bias package
# =======================================================

# arguments:
# name: name of the layer
# width: width of the binary
# nfrac: number of fractional bits
# weights_file: file containing weights
# biases_file: file containing biases
# num_weights: number of weights
# num_biases: number of biases

# creates file "[name]_[width]_[int].sv"
def generate_dense_pkg_file(name, dest_folder, width, nfrac, weights_file, biases_file, input_size, output_size):
    
    # determine the maximum and minimum values the fixed point number can represent
    max_val = (2**(width-1) - 1)/(2**nfrac)
    min_val = -2**(width-1)/(2**nfrac)

    # open source files
    src_weights_file = open(weights_file, 'r')
    src_biases_file = open(biases_file, 'r')
    
    package_name = get_pkg_name(name, width, nfrac)

    # make the final output files
    file_name = dest_folder + package_name + ".sv"
    dest_file = open(file_name, 'w')

    print(package_name)

    # first decalaration lines
    dest_file.write("// Width: " + str(width) + "\n// NFRAC: " + str(nfrac) + "\n")
    dest_file.write("package " + str(package_name) + ';\n\n')
    dest_file.write(f"localparam logic signed [{str(width-1)}:0] weights [{str(input_size)}][{str(output_size)}] = \'{{ \n")

    # convert weights (from floats) into binary
    for i in range(input_size):
        dest_file.write("{")
        for j in range(output_size):
            flt_num = float(src_weights_file.readline())
                        
            if (flt_num > max_val):
                flt_num = max_val
            elif (flt_num < min_val):
                flt_num = min_val

            binary_num = conv_to_str(flt_num, width, nfrac).replace('-', '')

            # overflow checking, now handled by converting extreme values to max/min values
            if (len(binary_num) < width):
                print("ERROR: binary number too large")
                print("binary number: " + binary_num)
                print("float number: " + str(flt_num))
                print("width: " + str(width))
                print("binary number length: " + str(len(binary_num)))
                print("from file: " + weights_file)
                raise ValueError("Binary number width inconsistent")


            dest_file.write(str(width) + "\'b" + binary_num)
            if (j != output_size - 1):
                dest_file.write(", ")
            else:
                dest_file.write("}")

        if (i != (input_size-1)): # no comma for last term
            # OPTIONAL: adds floating number as comment as reference
            dest_file.write(", " + "\n")
        else:
            dest_file.write("\n")                
    
    # switch to declaring biases
    dest_file.write("};\n\n")
    dest_file.write("localparam logic signed ["+ str(width-1) + ":0] bias [" + str(output_size) + "] = \'{\n")
    
    # convert bias (from floats) into binary
    for i in range(output_size):
        flt_num = float(src_biases_file.readline())
        
        # e.g., write line "17'b01011011011111001,"
        dest_file.write(str(width) + "\'b" + conv_to_str(flt_num, width, nfrac).replace('-', ''))
        
        if (i != (output_size-1)): # no comma for last term
            # OPTIONAL: adds floating number as comment as reference
            dest_file.write(",  // " + str(flt_num) + "\n")
        else:
            dest_file.write("   // " + str(flt_num) + "\n")
        
        
    # finish off declarations
    dest_file.write("};\nendpackage")
    
    
    dest_file.close()
    src_biases_file.close()
    src_weights_file.close()

In [6]:
def generate_gru_pkg_file(width, nfrac, input_size, output_size):

    # input size for outer candidate calculation (W) is input_size
    generate_dense_pkg_file("candidate_gate_W", "./new_weights/gru_weights_biases_pkgs/candidate_gate_weights_biases_pkgs/",
                            width, nfrac, "candidate_gate_weights_W.txt", "candidate_gate_biases_W.txt", input_size, output_size)

    # input size for inner candidate calculation (U) is size of h, which is output_size
    generate_dense_pkg_file("candidate_gate_U", "./new_weights/gru_weights_biases_pkgs/candidate_gate_weights_biases_pkgs/",
                            width, nfrac, "candidate_gate_weights_U.txt", "candidate_gate_biases_U.txt", output_size, output_size)
    
    # input size for each reset/update gates is input_size + h, which is input_size + output_size
    generate_dense_pkg_file("reset_gate", "./new_weights/gru_weights_biases_pkgs/reset_gate_weights_biases_pkgs/",
                            width, nfrac, "reset_gate_weights.txt", "reset_gate_biases.txt", input_size + output_size, output_size)
    generate_dense_pkg_file("update_gate", "./new_weights/gru_weights_biases_pkgs/update_gate_weights_biases_pkgs/",
                            width, nfrac, "update_gate_weights.txt", "update_gate_biases.txt", input_size + output_size, output_size)

In [8]:
# generate weight files for width 20, nfrac 10 (int 10)

width = 16
nfrac = 10
generate_gru_pkg_file(width, nfrac, INPUT_SIZE_GRU, OUTPUT_SIZE_GRU)

candidate_gate_W_16_6
candidate_gate_U_16_6
reset_gate_16_6
update_gate_16_6
